# Paradigm Core P1.2: Family-Conditioned Trusted Plasticity

This notebook inspects the recorded P1.2 benchmark. It does not rerun the full experiment by default.

Research links:

- `infinition/drift-contract` and the public Drift Contract preprint
- `infinition/z-manifold` and arXiv:2607.05300

The benchmark asks whether update trust becomes more useful when candidates share one parent lineage and are compared inside a declared behavior family.

In [ ]:
from pathlib import Path
import json
import pandas as pd

RESULT = Path("../results/core_p12/core_p12_family_trust.json")
if not RESULT.exists():
    RESULT = Path("results/core_p12/core_p12_family_trust.json")
result = json.loads(RESULT.read_text())
result.keys()

## Family-conditioned trust

Geometry is evaluated separately from semantic checks. A geometry pass is not a promotion decision.

In [ ]:
rows = []
for family, data in result["family_conditioned_trust"].items():
    for candidate_type in ["heldout_clean", "targeted_poison", "cross_family"]:
        m = data[candidate_type]
        rows.append({
            "family": family,
            "risk": data["risk"],
            "candidate": candidate_type,
            "parameter_acceptance": m["parameter_acceptance"],
            "behavior_acceptance": m["behavior_acceptance"],
            "joint_geometry": m["geometry_joint_acceptance"],
            "semantic": m["semantic_acceptance"],
            "full_manifest": m["full_manifest_acceptance"],
        })
pd.DataFrame(rows)

## Risk-conditioned epsilon

Same family, parent, data, minibatch seed, replay policy, and number of steps. Only epsilon changes.

In [ ]:
pd.DataFrame(result["risk_conditioned_epsilon"]["results"]).T

## Replay and bounded plasticity

Replay and Drift Contract address different failure modes. The comparison keeps accuracy, forgetting, and local drift separate.

In [ ]:
rows = []
for method, metrics in result["combined_baseline"].items():
    rows.append({
        "method": method,
        "current": metrics["final_current_accuracy"]["mean"],
        "mean_seen": metrics["final_mean_seen_accuracy"]["mean"],
        "worst_seen": metrics["final_worst_seen_accuracy"]["mean"],
        "forgetting": metrics["final_mean_forgetting"]["mean"],
        "step_drift": metrics["mean_max_step_drift"]["mean"],
    })
pd.DataFrame(rows).set_index("method")

## Interpretation

The recorded result supports a narrower design than a global trusted update manifold:

1. compare candidates within one parameter lineage
2. condition geometry on a validated behavior family
3. measure parameter and behavior deltas independently
4. keep semantic subgroup checks mandatory
5. route unknown families to deliberation rather than treating them as invalid
6. use epsilon as a plasticity budget, not a confidence score

The next milestone is P1.3, which connects these signals to the persistent active/candidate lifecycle.